In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from tqdm import tqdm
import pandas as pd
import time
import numpy as np

##Import data
csvfile = 'marketing_campaign_cleaned_NODUMMIES'
df = pd.read_csv(csvfile)

#Synthcity
import sys
import warnings

warnings.filterwarnings("ignore")

import synthcity.logger as log
from synthcity.plugins import Plugins
from synthcity.plugins.core.dataloader import GenericDataLoader

log.add(sink=sys.stderr, level="INFO")

#Fit GAN using ALL training data
from synthcity.plugins import Plugins

syn_model_full = Plugins().get("ctgan")

##########################FUNCTIONS#################################

def dummify_columns(df):
    # Dummify marital and educational
    dummify_marital = pd.get_dummies(df['Marital_Status'],prefix='marital')
    df = pd.concat([df, dummify_marital],axis=1)

    dummify_edu = pd.get_dummies(df['Education'],prefix='education')
    df = pd.concat([df, dummify_edu], axis=1)

    # Drop transformed cols
    df.drop(columns=['Marital_Status', 'Education'], inplace=True)
    
    return df

def train_and_evaluate(df_train, df_test):
    # Split into features and target for train and test datasets
    X_train = df_train.drop('Response', axis=1)
    y_train = df_train['Response']
    X_test = df_test.drop('Response', axis=1)
    y_test = df_test['Response']

        
    # Identify overlapping columns in train and test: note that the train set is only small (7.5p), columns may be missing
    # if we add only a little bit of synth data, there is a chance that same column is missing in synth data
    common_columns = set(X_train.columns).intersection(X_test.columns)
    
    # Keep only common columns in train and test
    X_train = X_train[common_columns]
    X_test = X_test[common_columns]
    
    # Train a Random Forest model
    model = RandomForestClassifier(n_estimators = 500, max_depth = None, 
                                   max_features = 'auto', criterion = 'gini', min_samples_split = 2)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    report = classification_report(y_test, y_pred, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store metrics
    results = {
        'accuracy': report['accuracy'],
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'auc-roc': roc_auc
    }

    return results





<stdin>:1:10: fatal error: 'omp.h' file not found
#include <omp.h>
         ^~~~~~~
1 error generated.


[KeOps] Warning : omp.h header is not in the path, disabling OpenMP.
[KeOps] Warning : Cuda libraries were not detected on the system ; using cpu only mode
2023-08-07 21:47:54,250 - Created a temporary directory at /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmph05mopf9
2023-08-07 21:47:54,250 - Writing /var/folders/8w/flfvck1j6m77x3j6jf0384bw0000gn/T/tmph05mopf9/_remote_module_non_scriptable.py


In [2]:
# THE other FASST extrinisc15 dataset had random state = 4

countlen = len(df)*0.7*0.925

syn_sizes = [1, 0.5*countlen, 1*countlen,  3*countlen,  5*countlen, 
            8*countlen, 12*countlen, 18*countlen, 32*countlen, 48*countlen, 64*countlen]

n_iterations = 300 # Number of bootstrapping iterations

# Placeholder for the results
results = []

# Bootstrap iteration loop
for i in range(n_iterations):
    # Resample entire dataset
    start_time = time.time()

    df_resampled = resample(df, replace=True)

    
    
    
    
    # Perform train/test split
    df_train_main, df_test = train_test_split(df_resampled, test_size=0.3, 
                                         stratify=df_resampled['Response'], random_state=i)

    df_train_1, df_train_2 = train_test_split(df_train_main, test_size=0.075, random_state=i, 
                                              stratify=df_train_main['Response'])

    loader = GenericDataLoader(df_train_1, target_column='Response')
    syn_model = Plugins().get('ctgan')
    syn_model.fit(loader,cond=df_train_1['Response'].to_frame())
    # Loop through the different synthetic set sizes
    for size in syn_sizes:
        # Generate synthetic set of the current size based on resampled training data
        #syn_set = syn_model.generate(count=size,random_state=i*124).dataframe()
        syn_set = syn_model.generate(count=size,random_state=i,cond=np.random.permutation([1]*int(round(df['Response'].mean()*size)) + [0]*int(size-round(df['Response'].mean()*size)))).dataframe()

        print(syn_set['Response'].mean() * 100)

        # Add synthetic data to resampled training data
        df_train_combined = pd.concat([df_train_2, syn_set], axis=0)

        # Dummify train and test datasets
        df_train_combined = dummify_columns(df_train_combined)
        df_test_dummified = dummify_columns(df_test.copy())  # .copy() to avoid SettingWithCopyWarning

        # Train and evaluate
        metrics = train_and_evaluate(df_train_combined, df_test_dummified)

        # Store the results with additional information
        metrics['syn_size'] = size
        metrics['iteration'] = i
        results.append(metrics)

    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time: {execution_time} seconds")
    print(f"Iteration: {i} " )
    
# Convert to DataFrame
results_exc_df_1 = pd.DataFrame(results)

 20%|███████▉                                | 399/2000 [01:48<07:13,  3.69it/s]


0.0
14.586255259467041
15.206727400140155
14.989493345785665
15.242364808069485
15.016198231328254
15.176278309596078
15.171796567959841
15.155295817190886
15.115787016094906
15.17264021887825
Time: 319.09756803512573 seconds
Iteration: 0 


 45%|█████████████████▉                      | 899/2000 [04:18<05:17,  3.47it/s]


0.0
14.586255259467041
14.926419060967064
14.639271538641138
14.681983748949284
14.919884423430524
14.977819285547515
14.99669247830655
14.883883818153961
14.888152806758987
15.00410396716826
Time: 465.857971906662 seconds
Iteration: 1 


 60%|███████████████████████▍               | 1199/2000 [05:53<03:56,  3.39it/s]


...

 25%|█████████▉                              | 499/2000 [02:29<07:29,  3.34it/s]


0.0
15.147265077138849
15.487035739313246
15.176278309596078
15.214345755113476
15.033709832764206
15.211300490310531
15.121210942060001
15.350099590693194
15.197501860471904
15.20437756497948
Time: 349.367760181427 seconds
Iteration: 125 


 32%|████████████▉                           | 649/2000 [03:19<06:54,  3.26it/s]


0.0
14.866760168302944
14.926419060967064
14.989493345785665
15.270383861025497
15.375186060765255
15.252159701144057
15.253511809798045
15.251603300720118


KeyboardInterrupt: 

In [ ]:
results_exc_df_1.to_clipboard()